# SOXX 前日价格/NAV 偏离与五日现金优势：条件性必要信息检验

冻结正向秩假说、完整日历和一次区块重抽；历史首次发布时点未认证，禁止据此建账户。


In [1]:
import os,sys,json,hashlib,tarfile,tempfile,subprocess,shlex
from pathlib import Path
import numpy as np,pandas as pd,qlib
from scipy.stats import spearmanr
from qlib.data import D
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.workflow import R
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import archive_nas_recorder,_settings
card=root/'research/stage1_soxx_nav_premium_cash_h5_necessary_information_20261023';plan=json.loads((card/'plan.json').read_text());src=plan['sources'];sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('external_requests','new_model_fits','new_native_accounts'))
a=root/src['premium_source_card'];b=root/src['cash_label_card'];sa=json.loads((a/'summary.json').read_text());sb=json.loads((b/'summary.json').read_text())
assert sha(a/'summary.json')==src['premium_source_summary_sha256'] and sha(b/'summary.json')==src['cash_label_summary_sha256']
assert sa['recorder_id']==src['premium_source_recorder_id'] and sb['recorder_id']==src['cash_label_recorder_id']
assert sa['nas_evidence']['sha256']==src['premium_source_NAS_sha256'] and sb['nas_evidence']['sha256']==src['cash_label_NAS_sha256']
assert sha(a/'qlib_recorder'/src['premium_source_experiment_id']/src['premium_source_recorder_id']/'artifacts/analysis/source_frame')==src['premium_source_frame_sha256']
assert sha(b/'qlib_recorder'/src['cash_label_experiment_id']/src['cash_label_recorder_id']/'artifacts/analysis/daily')==src['cash_label_daily_sha256']
assert sha(root/src['Qlib_provider']/'qlib.yaml')==src['Qlib_config_sha256']
os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(root/src['Qlib_provider']),region='us',kernels=1)
with R.uri_context(sa['recorder_uri']):source_frame=R.get_recorder(recorder_id=src['premium_source_recorder_id'],experiment_id=src['premium_source_experiment_id']).load_object('analysis/source_frame')
with R.uri_context(sb['recorder_uri']):label=R.get_recorder(recorder_id=src['cash_label_recorder_id'],experiment_id=src['cash_label_experiment_id']).load_object('analysis/daily')
cal=pd.DatetimeIndex(D.calendar(start_time=plan['cohort']['start'],end_time=plan['cohort']['end'],freq='day'))
assert len(cal)==plan['cohort']['full_calendar_rows']==len(label)==944 and cal.equals(pd.DatetimeIndex(pd.to_datetime(label.signal_date)))
prem=source_frame['price_to_NAV']-1
lagged=prem.shift(1).reindex(cal)
source_date=pd.Series(source_frame.index,index=source_frame.index).shift(1).reindex(cal)
assert lagged.notna().all() and source_date.notna().all() and (source_date<cal).all()
assert np.isfinite(lagged).all() and np.isfinite(label.cash_advantage_h5.iloc[:-4]).all()
mature=label.cash_advantage_h5.notna() & label.label_available_h5.notna();assert mature.sum()==plan['cohort']['mature_H5_rows']==940 and label.index[~mature].tolist()==[940,941,942,943]
assert (pd.to_datetime(label.loc[mature,'label_available_h5'])>pd.to_datetime(label.loc[mature,'trade_date'])).all()
d=pd.DataFrame({'signal_date':cal,'trade_date':pd.to_datetime(label.trade_date),'source_date':source_date.to_numpy(),'prior_session_premium':lagged.to_numpy(dtype=float),'cash_advantage_h5':label.cash_advantage_h5.to_numpy(dtype=float),'H5_mature':mature.to_numpy(dtype=bool),'H5_available_at':label.label_available_h5})
static=pd.concat({'feature':d[['prior_session_premium']],'label':d[['cash_advantage_h5']],'meta':d[['H5_mature']]},axis=1)
static.index=pd.MultiIndex.from_arrays([cal,['SOXX']*len(cal)],names=['datetime','instrument']);ds=DatasetH(handler=DataHandlerLP.from_df(static),segments={'held':(plan['cohort']['start'],plan['cohort']['end'])});assert len(ds.prepare('held',col_set=['feature','label','meta']))==944
rho=lambda z:float(spearmanr(z.prior_session_premium,z.cash_advantage_h5).statistic) if len(z)>10 and z.prior_session_premium.nunique()>1 else None
h=d.loc[d.H5_mature];primary=rho(h);assert primary is not None
half1=h.loc[h.signal_date<pd.Timestamp('2025-01-01')];half2=h.loc[h.signal_date>=pd.Timestamp('2025-01-01')];halves=[rho(half1),rho(half2)];assert min(len(half1),len(half2))>300
annual={str(y):{'rows':len(g),'spearman':rho(g),'mean_premium':float(g.prior_session_premium.mean()),'mean_cash_advantage_h5':float(g.cash_advantage_h5.mean())} for y,g in h.groupby(h.signal_date.dt.year)}
quarters={str(q):{'rows':len(g),'spearman':rho(g),'mean_premium':float(g.prior_session_premium.mean()),'mean_cash_advantage_h5':float(g.cash_advantage_h5.mean())} for q,g in h.groupby(h.signal_date.dt.to_period('Q'))};assert len(quarters)==16
cfg=plan['statistics']['uncertainty'];tracking=card/'qlib_recorder';state={'plan_sha256':sha(card/'plan.json'),'bootstrap_attempted':True,'bootstrap_draw_budget':cfg['draws'],'model_fits':0,'native_accounts':0};(card/'run_state.json').write_text(json.dumps(state,indent=2)+'\n')
rng=np.random.default_rng(cfg['seed']);n=len(d);rep=np.empty(cfg['draws'])
for i in range(cfg['draws']):
 idx=np.empty(n,dtype=np.int32);idx[0]=rng.integers(n)
 for j in range(1,n):idx[j]=rng.integers(n) if rng.random()<1/cfg['mean_block_sessions'] else (idx[j-1]+1)%n
 sample=d.iloc[idx];sample=sample.loc[sample.H5_mature & sample.cash_advantage_h5.notna()];rep[i]=rho(sample)
assert np.isfinite(rep).all();ci=np.quantile(rep,cfg['two_sided_descriptive_quantiles']);support=bool(primary>0 and all(x is not None and x>0 for x in halves) and all(annual[str(y)]['spearman']>0 for y in plan['cohort']['complete_years']) and ci[0]>0)
report={'decision':'conditional_prior_premium_cash_H5_necessary_support_development_only' if support else 'closed_fixed_prior_premium_cash_H5_necessary_information_failed','primary_H5_spearman':primary,'held_halves_H5_spearman':halves,'complete_2024_2025_H5_spearman':{str(y):annual[str(y)]['spearman'] for y in (2024,2025)},'descriptive_95pct_full_calendar_block_interval':[float(ci[0]),float(ci[1])],'necessary_support':support,'annual':annual,'quarters':quarters,'calendar_rows':944,'mature_H5_rows':940,'bootstrap_replicates':len(rep),'new_model_fits':0,'new_native_accounts':0,'historical_independence':False,'historical_PIT_qualified':False,'project_wide_search_corrected':False}
assert not tracking.exists()
with R.start(experiment_name=plan['experiment'],uri=str(tracking)):
 rec=R.get_recorder();rid=rec.id;eid=str(rec.experiment_id);ds.config(dump_all=True,recursive=True);rec.save_objects(artifact_path='inputs',plan=plan,dataset=ds);rec.save_objects(artifact_path='analysis',matched_daily=d,report=report,bootstrap_replicates=rep);rec.log_metrics(primary_H5_spearman=primary,block_lower=float(ci[0]),necessary_support=int(support),model_fits=0,native_accounts=0)
(card/'result.json').write_text(json.dumps(report,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':report['decision'],'objective_achieved':False,'plan_sha256':sha(card/'plan.json'),'result':'result.json','recorder_id':rid,'experiment_id':eid,'recorder_uri':str(tracking),'new_model_fits':0,'new_native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Historical Yahoo/issuer NAV first-publication and revisions unqualified; current revised history only.','Reused 2023-2026 held history is development feedback, and 2000-block interval not project-wide corrected.','Necessary rank association is not incremental original39 information, full Stage1 account, fees or independent forward evidence.']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
settings=_settings(root);binding=card/'nas_binding.json';binding.write_text(json.dumps({'schema_version':'recorder_evidence_binding_v1','authority':'yanxuan_nas','source_id':plan['experiment'],'ssh':settings['ssh'],'source_plan':'plan.json'},indent=2)+'\n')
e=archive_nas_recorder(binding,workspace_root=root,card=card,recorder_id=rid,experiment_id=eid)
with tempfile.TemporaryDirectory(prefix='soxx-premium-h5-') as tmp:
 archive=Path(tmp)/'recorder.tar.gz'
 with archive.open('wb') as out:run=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(e['nas_path'])],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert run.returncode==0 and sha(archive)==e['sha256']
 with tarfile.open(archive,'r:gz') as z:assert any(m.name.endswith('/artifacts/analysis/report') for m in z.getmembers())
summary['nas_evidence']=e;summary['nas_readback']={'verified':True,'sha256':e['sha256']};(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(json.dumps({'primary':primary,'halves':halves,'complete_years':report['complete_2024_2025_H5_spearman'],'ci':report['descriptive_95pct_full_calendar_block_interval'],'support':support,'recorder_id':rid},indent=2))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[23699:MainThread](2026-10-23 07:51:14,454) INFO - qlib.Initialization - [config.py:453] - default_conf: client.


[23699:MainThread](2026-10-23 07:51:14,456) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.


[23699:MainThread](2026-10-23 07:51:14,456) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


...

/Users/rziiiii/Library/Python/3.14/lib/python/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[23699:MainThread](2026-10-23 07:51:16,051) WARNING - qlib.workflow - [expm.py:230] - No valid experiment found. Create a new experiment with name stage1_soxx_nav_premium_cash_h5_necessary_information_20261023.


[23699:MainThread](2026-10-23 07:51:16,053) INFO - qlib.workflow - [exp.py:258] - Experiment 593350234683233200 starts running ...


[23699:MainThread](2026-10-23 07:51:16,122) INFO - qlib.workflow - [recorder.py:345] - Recorder 9a48dbc2fa994f05a42832eee0a6dddd starts running under Experiment 593350234683233200 ...


[23699:MainThread](2026-10-23 07:51:17,252) INFO - qlib.timer - [log.py:127] - Time cost: 0.002s | waiting `async_log` Done


{
  "primary": 0.019706768399538513,
  "halves": [
    0.04131638349401171,
    -0.006083242011750618
  ],
  "complete_years": {
    "2024": 0.04256566806367603,
    "2025": -0.02947055152882446
  },
  "ci": [
    -0.03837375896088763,
    0.07372372515474371
  ],
  "support": false,
  "recorder_id": "9a48dbc2fa994f05a42832eee0a6dddd"
}


# SOXX 前日价格/NAV 偏离与五日现金优势：必要信息检验

保持冻结的原 Stage1 日历与五日 SOXX 现金优势标签。信号日只使用**前一个 Qlib 交易日**的 Yahoo 原始 SOXX 收盘价 ÷ iShares 官方 NAV − 1；不使用信号日收盘、调整价、反号或收益分组。944 个完整评价日全部有特征，五日标签成熟 940 日，末尾 4 日保留未知。2023–2026 已反复使用，只能作为条件性开发反馈；历史 NAV 和 Yahoo 的首次公布时刻及当时版本仍未认证。

预先规定的正向秩关系没有获得必要支持：全期 Spearman **+0.01971**；前后固定半段 **+0.04132 / −0.00608**；完整 2024、2025 年 **+0.04257 / −0.02947**。一次按完整 944 日历、平均区块 63 日、2,000 次的 stationary bootstrap 描述性 95% 区间为 **[−0.03837, +0.07372]**，下界不大于零。四个年度、16 个季度和全部成熟日均保存在原生 Qlib Recorder，未从较好季度选规则。

Qlib Recorder `9a48dbc2fa994f05a42832eee0a6dddd` 的 944 行 DatasetH、940 个成熟标签、2,000 个重抽样和报告已读回；NAS 归档按哈希冷读通过。0 新拟合、0 账户。这个单坐标必要检验既不是相对原 39 项的增量模型证据，也不是完整 Stage1 0bp Tiered 账户。

**决定：`closed_fixed_prior_premium_cash_H5_necessary_information_failed`，不推广。** 不在已用历史上改符号、前日滞后、五日目标、阈值或季度来挽救此版。下一机制须提供与这条 ETF 价格/NAV 偏离不同的决策前信息，先过来源时点与增量信息门，再接受完整账户和 SOXX 硬门。
